In [18]:
# 다음 리뷰데이터 로드
# 평점 최저 최고 점수 확인한 후 분포도 확인해서
# 긍정과 부정의 threshold를 찾아서 target컬럼에 0(긍정), 1(부정)로 추가 컬럼 생성

In [19]:
import pandas as pd
df = pd.read_csv("../day02/daum_movie_review.csv")
df.head()

,review,rating,date,title
0,돈 들인건 티가 나지만 보는 내내 하품만,1,2018.10.29,인피니티 워
1,몰입할수밖에 없다. 어렵게 생각할 필요없다. 내가 전투에 참여한듯 손에 땀이남.,10,2018.10.26,인피니티 워
2,이전 작품에 비해 더 화려하고 스케일도 커졌지만.... 전국 맛집의 음식들을 한데 ...,8,2018.10.24,인피니티 워
3,이 정도면 볼만하다고 할 수 있음!,8,2018.10.22,인피니티 워
4,재미있다,10,2018.10.20,인피니티 워


In [23]:
df['rating'].max(), df['rating'].min()

(10, 0)

In [24]:
df['target'] = df['rating'].apply(lambda x : 0 if x > 5 else 1)

In [25]:
# 전처리
import re
from konlpy.tag import Okt
from sklearn.feature_extraction.text import TfidfVectorizer
df['cleaned_review'] = df['review'].apply(lambda x : re.sub(r'[^가-힣\s]','',x) )
# 토큰화(한글은 형태소)
okt = Okt()
def kor_tokenize(text):
    return [word for word, _ in okt.pos(text, stem=True)]

tfidf = TfidfVectorizer(tokenizer=kor_tokenize,max_features=5000)
x_tfidf = tfidf.fit_transform(df['cleaned_review']).toarray()
y = df['target'].values

In [30]:
%pip uninstall torch torchvision torchaudio

^C
Note: you may need to restart the kernel to use updated packages.


In [22]:
from torch.utils.data import Dataset, DataLoader
import torch
class TfidfDataset(Dataset):
    def __init__(self,vectors, labels):
        self.vectors = torch.FloatTensor(vectors)
        self.labels = torch.FloatTensor(labels)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        return self.vectors[index], self.labels[index]

AttributeError: partially initialized module 'torch' from 'c:\Users\Playdata\miniconda3\Lib\site-packages\torch\__init__.py' has no attribute 'nn' (most likely due to a circular import)

In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x_tfidf,y,random_state=42,test_size=0.2)
train_loader = DataLoader(TfidfDataset(x_train,y_train), batch_size=32,shuffle=True)
test_loader = DataLoader(TfidfDataset(x_test, y_test), batch_size=32, shuffle=False)

In [ ]:
import torch.nn as nn
class TfidfMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim,hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden_dim,1),
        )
    def forward(self, x):
        return self.network(x)    

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
from torch.optim import Adam
x_train.shape[1]
model = TfidfMLP(input_dim = x_train.shape[1], hidden_dim = 64).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = Adam(model.parameters(), lr=1e-3)

In [ ]:
%pip install ipywidgets

In [ ]:
from tqdm import tqdm
epochs = 10
pbar = tqdm(range(epochs), desc="Training")
for epoch in pbar:
    model.train()
    local_loss = 0.0
    for vecs, labels in train_loader:
        vecs, labels = vecs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(vecs).squeeze(1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        local_loss += loss.item()
    avg_loss = local_loss / len(train_loader)
    pbar.set_postfix(
        loss=f"{avg_loss:.4f}"
    )   

In [ ]:
# 예측... 정확도
model.eval()
correct  = 0.0; total = 0.0
with torch.no_grad():
    for vecs, labels in test_loader:
        vecs,labels = vecs.to(device), labels.to(device)
        preds = (torch.sigmoid(model(vecs)).squeeze(1) > 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.shape[0]
    print(f'test accuracy : {(correct / total):.4f}')